# H2 Morocco – Gesamtlauf: MCA + Kostenanalyse + Vergleich

**Status: in Gebrauch.** Teil B rechnet die ganze Pipeline aus einem leeren `data/interim/` (Referenzlauf 05.11.2026, etwa 20 Minuten). Teil A sammelt, was sich noch verbessern ließe.

- **Teil A** (Umbau) – was an den Skripten geändert werden müsste, in kleinen Schritten.
- **Teil B** (Lauf) – startet alle Teilskripte als eigene Prozesse (außer 00 und 01, siehe unten). Zwei Vergleichsfolien:
  `compare_mca_runs.py` (MCA gegenüber dem letzten MCA-Lauf, Laufordner `*_mca`) und zum Schluss
  `compare_mca_cost.py` (MCA ↔ Kosten mit Δ gegenüber dem Bezugslauf `REFERENCE`).
- **Teil C** (Vergleich) – läuft schon jetzt mit den vorhandenen Ergebnissen:
  MCA aus `results/mca/`, Kosten aus dem neuesten Laufordner `results/runs/*_cost`.

Ohne Oberfläche: `jupyter nbconvert --to notebook --execute main.ipynb --output /tmp/main_out.ipynb`

## Teil A – Was umgebaut werden müsste

### Ist-Zustand

```
MCA/   Kriterien-Skripte (7219-Zellen-Schleifen, teils sehr lang)
         -> data/interim/mca_criteria/results_*.csv
       mca_maps.py   Rechnung + Karten in einem Skript, Gewichte fest im Code
         -> results/mca/results_mca_morocco_2025_2050.csv      (wird überschrieben)
       compare_mca_runs.py -> results/runs/<datum>_<commit>_mca/  (Kopie von Ergebnis, Kriterien, Gewichten)

Cost_analysis/  01 -> 02 (Gurobi) -> 03 -> 04 -> 05
         04 -> results/runs/<datum>_<commit>_cost/            (eigener Laufordner, Übersichtsbild)
```

Beide Analysen laufen getrennt und dokumentieren sich unterschiedlich.
Später lässt sich deshalb nicht sagen, **welcher MCA-Stand zu welchem Kostenlauf gehört.**

### Umbau-Schritte (je ein Schritt = eine Idee)

| # | Schritt | Warum | Aufwand |
|---|---|---|---|
| 1 | **Gemeinsamer Laufordner**: `main` legt `results/runs/<datum>_<commit>_main/` an und übergibt ihn den Skripten (`--run-dir` oder Umgebungsvariable `H2M_RUN_DIR`). 04 legt nur dann einen eigenen Ordner an, wenn keiner übergeben wird. | MCA, Kosten und Vergleich eines Laufs liegen zusammen, mit einem Commit und einem Parameter-Snapshot. | klein |
| 2 | **`mca_maps.py` teilen**: Rechnung (z. B. `MCA/06_mca_score.py` oder Funktion `h2morocco/mca.py::mca_score()`) getrennt von den Karten. Die Rechnung schreibt in den Laufordner, *auch die skalierten Einzelkriterien je Zelle*. | Die Rechnung dauert Sekunden und braucht keinen CARTO-Key. Die Einzelkriterien braucht der Vergleich je Kriterium (C.4). | klein–mittel |
| 3 | **Teure Kriterien nicht jedes Mal rechnen**: In `STEPS` sind sie standardmäßig aus. Später ließe sich das automatisch prüfen, z. B. ob eine Eingabedatei neuer ist als die Ergebnis-CSV (wie bei `make`). | `accesibility.py` und die anderen Kriterien-Schleifen dauern lange, die Aggregation nur Sekunden. | klein (manuell) / mittel (automatisch) |
| 4 | **MCA-Gewichte in eine Parameterdatei** (`data/model_parameters/mca_weights.csv`, Snapshot im Laufordner). Das ist bewusst noch zurückgestellt. | Gewichtsänderungen tauchen dann im Übersichtsbild als „geänderte Parameter“ auf, genau wie Kostenparameter. | klein |
| 5 | ~~**Ein Übersichtsbild für den Gesamtlauf**~~ – erledigt: `compare_mca_cost.py` (letzter Schritt in `STEPS`). | Pro Lauf eine Folie: Kosten, MCA, Übereinstimmung, Änderungen gegenüber dem Bezug. | erledigt |
| 6 | **Funktionen statt Skripte** (optional, später): `run_mca(params) -> DataFrame`, `run_cost(params) -> DataFrame` im Paket `h2morocco`. | Ermöglicht Sensitivitätsläufe (z. B. Monte-Carlo über Gewichte oder Zins), ohne Dateien zu schreiben. Nur für die schnellen Schritte (MCA-Aggregation, 04) sinnvoll. | groß |
| 7 | **Laufordner für die MCA** – teilweise erledigt: `compare_mca_runs.py` legt nach `mca_maps.py` einen Ordner `*_mca` an (Ergebnis, Kriterien-CSVs, Gewichte aus `mca_maps.py`, Prüfsummen) und zeichnet eine Vergleichsfolie. Die skalierten Einzelkriterien fehlen noch (Schritt 2). | Jeder MCA-Stand ist festgehalten und mit dem vorherigen vergleichbar. | teilweise |

**Empfehlung:** Schritte 1 → 2 als Nächstes. Danach ist `main` ein vollständiger, dokumentierter Lauf.
Schritt 6 erst, wenn Sensitivitätsanalysen konkret geplant sind.

### Inhaltliche Punkte, die der Vergleich sichtbar macht (keine Code-Fragen)

- **Die beiden Analysen sind nicht unabhängig.** PV/Wind-Ressource und Wasser stecken in beiden. Eine hohe Übereinstimmung ist deshalb zum Teil Konstruktion, kein Befund. C.4 zeigt, welche Kriterien die Übereinstimmung tragen.
- **No-Go-Zonen** wirken nur in der MCA (Fehlerprotokoll F-25). Der Vergleich rechnet deshalb nur auf zulässigen Zellen.
- **Unterschiedliche Richtung**: Ein hoher MCA-Score ist gut, hohe Kosten sind schlecht. Ein Spearman-ρ < 0 bedeutet also „stimmen überein“.
- Die MCA ist **relativ** (Min-Max-Skalierung je Jahr), die Kosten sind **absolut**. Vergleichbar sind deshalb Ränge, nicht Werte.

## Teil B – Lauf

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from scipy.stats import spearmanr

from h2morocco.runs import ROOT, latest_run_dir, new_run_dir

YEARS = [2025, 2050]
NOTE = ''          # kurze Notiz zum Lauf, landet in run_info.json
SAVE = False       # True: Vergleich in einen neuen Laufordner results/runs/<...>_compare schreiben

### Schritte

Reihenfolge = Abhängigkeitsreihenfolge, nicht umsortieren. Ist ein Schritt aus, nutzen die folgenden Schritte die vorhandenen Dateien.

In [ ]:
# Bezug für die Δ-Werte auf der Vergleichsfolie (None = letzter Vergleichslauf)
REFERENCE = 'results/runs/2026-11-05_1041_original_compare'   # veröffentlichtes Original

STEPS = {
    # --- MCA: Vorverarbeitung und Kriterien (Schleifen über 7219 Zellen, teils sehr lang) ---
    'MCA/Grid_morocco/grid_morocco.py': True,    # Raster roh + auf Landesgrenze zugeschnitten (grid_morocco_clear)
    'MCA/forecast_landuse.py':       True,    # Landnutzung 2050
    'MCA/water_exploitable_gw.py':   True,
    'MCA/ki_generiert/flow_direction_grid.py': True,    # Zonensumme Fließrichtung (Fehlerprotokoll Nr. 110), ca. 10 s, 3,5 GB RAM
    'MCA/water_exploitable_sw.py':   True,
    'MCA/ki_generiert/irrigation_grid.py': True,    # Bewässerungsfläche je Zelle aus FAO GMIA (für Agrarwasser)
    'MCA/water_consumption.py':      True,
    'MCA/water_net.py':              True,    # Wasser 2025
    'MCA/water_forecast.py':         True,    # Wasser 2050 (Klimadaten 2050: Original aus Sciebo, data/README.md)
    'MCA/ki_generiert/pv_yield_polygons.py': True,    # PV-Polygone aus Global Solar Atlas
    'MCA/pv_analysis.py':            True,
    'MCA/ki_generiert/wind_flh_grid.py': True,  # Wind-Volllaststunden aus Global Wind Atlas (IEC III)
    'MCA/wind_analysis.py':          True,
    'MCA/accesibility.py':           True,    # sehr lang
    'MCA/agriculture.py':            True,
    'MCA/urban_zone.py':             True,
    'MCA/rural_zone.py':             True,
    'MCA/industrie.py':              True,
    'MCA/non_conflict.py':           True,
    'MCA/ki_generiert/topography_grid.py': True,    # Höhenspanne je Zelle aus Copernicus DEM
    'MCA/no_go_zones.py':            True,
    # --- MCA: Aggregation und Karten. Schreiben noch fest nach results/mca/ und results/figures/mca/ ---
    'MCA/mca_maps.py':               True,
    'MCA/MapScripts/landuse_map.py': True,
    'MCA/MapScripts/water_maps.py': True,
    'MCA/nogo_maps.py':              True,
    'MCA/MapScripts/re_potential_map.py': True,
    'MCA/MapScripts/infra_map.py': True,
    # --- MCA-Vergleich (optional): Laufordner *_mca + Folie overview_mca.png ggü. letztem MCA-Lauf ---
    'compare_mca_runs.py':           True,
    # --- Kosten ---
    # 00_era5_download.py und 01_re_data.py fehlen bewusst: 00 lädt ERA5 (Stunden, CDS-Warteschlange),
    # 01 lädt nur Punkt 0 nach und hängt ihn doppelt an pv_wind_electricity.csv an (Fehlerprotokoll Nr. 10).
    'Cost_analysis/ki_generiert/01b_flh_re.py':       True,   # Volllaststunden der Referenzpunkte aus den Profilen in data/raw/
    'Cost_analysis/ki_generiert/01c_cell_flh_re.py': True, # PV/Wind je Zelle aus den MCA-Kriterien, für die Zuordnung in 04
    'Cost_analysis/02_FLH_el.py':                    True,   # Gurobi, je Szenariojahr, ca. 8 min
    'Cost_analysis/03_assignment_re_data.py':        True,
    'Cost_analysis/04_cost_analysis_h2_morocco.py':  True,   # ca. 30–40 s, legt Laufordner *_cost an
    'Cost_analysis/05_results_cost.py':              True,   # Karten in diesen Kostenlauf, CARTO-Key
    # --- Vergleich MCA ↔ Kosten: Übersichtsfolie in neuem Laufordner *_compare ---
    'compare_mca_cost.py':           True,
}

# Zusätzliche Argumente je Skript
ARGS = {
    'Cost_analysis/04_cost_analysis_h2_morocco.py': ['--note', NOTE],
    'compare_mca_runs.py': ['--note', NOTE],
    'compare_mca_cost.py': ['--note', NOTE] + (['--reference', REFERENCE] if REFERENCE else []),
}

In [ ]:
# Agg: plt.show() in den Skripten blockiert nicht, Karten werden nur gespeichert.
ENV = {**os.environ, 'MPLBACKEND': 'Agg',
       'PYTHONWARNINGS': 'ignore:FigureCanvasAgg is non-interactive:UserWarning'}
LOG = ROOT / 'results/runs/main_last.log'   # Ausgabe aller Schritte, auch während des Laufs lesbar
LOG.write_text(f'main.ipynb gestartet {datetime.now():%Y-%m-%d %H:%M:%S}\n', encoding='utf-8')


def run(script):
    # Startet ein Skript als eigenen Prozess, zeigt die Ausgabe live, bricht bei Fehler ab.
    start = time.perf_counter()
    cmd = [sys.executable, str(ROOT / script), *ARGS.get(script, [])]
    with open(LOG, 'a', encoding='utf-8') as log:
        header = f'\n=== {script}  ({datetime.now():%H:%M:%S})'
        print(header, flush=True)
        log.write(header + '\n')
        with subprocess.Popen(cmd, env=ENV, cwd=ROOT, text=True,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
        duration = time.perf_counter() - start
        log.write(f'--- {script}: Code {proc.returncode}, {duration:.0f} s\n')
    if proc.returncode != 0:
        raise RuntimeError(f'{script} ist mit Code {proc.returncode} abgebrochen ({duration:.0f} s)')
    return duration


durations = {script: run(script) for script, enabled in STEPS.items() if enabled}
for script in STEPS:
    status = f'{durations[script]:7.0f} s' if script in durations else '  übersprungen'
    print(f'{status}  {script}')

In [ ]:
# Übersichtsfolie des Vergleichs anzeigen
from IPython.display import Image, display

compare_dir = latest_run_dir('compare')
print(compare_dir.relative_to(ROOT))
display(Image(filename=compare_dir / 'overview.png'))

# Übersichtsfolie des MCA-Vergleichs (falls compare_mca_runs.py lief)
mca_dir = latest_run_dir('mca')
if mca_dir is not None and (mca_dir / 'overview_mca.png').exists():
    print(mca_dir.relative_to(ROOT))
    display(Image(filename=mca_dir / 'overview_mca.png'))

## Teil C – Vergleich MCA ↔ Kosten

### C.1 Ergebnisse laden und Herkunft prüfen

In [ ]:
MCA_CSV  = ROOT / 'results/mca/results_mca_morocco_2025_2050.csv'   # nach Umbau 1+2: <Laufordner>/results_mca.csv
COST_RUN = latest_run_dir('cost')                                     # oder fest: ROOT / 'results/runs/<ordner>'

gdf_grid = gpd.read_file(ROOT / 'data/interim/grid/grid_morocco_clear.shp').to_crs('EPSG:32629')
df_mca   = pd.read_csv(MCA_CSV)
df_cost  = {year: pd.read_csv(COST_RUN / f'results_cost_{year}.csv') for year in YEARS}
allowed  = pd.read_csv(ROOT / 'data/interim/mca_criteria/results_nogo_zones.csv')['NoGo'] == 1

# Alle Tabellen sind zeilengleich mit dem Raster (Zeile i = Zelle i). Das muss stimmen, sonst ist alles verschoben.
for name, n in [('MCA', len(df_mca)), ('No-Go', len(allowed))] + [(f'Kosten {y}', len(df_cost[y])) for y in YEARS]:
    assert n == len(gdf_grid), f'{name}: {n} Zeilen, Raster hat {len(gdf_grid)}'

mca  = {year: df_mca[f'MCA_Morocco_{year}'] for year in YEARS}
lcoh = {year: df_cost[year][f'H2_Price_{year} [EUR/MWh_h2]'] for year in YEARS}

# Herkunft: Genau das fehlt heute für die MCA (kein Commit, kein Parameter-Snapshot)
cost_info = json.loads((COST_RUN / 'run_info.json').read_text(encoding='utf-8'))
print(f'Kosten: {COST_RUN.name}  (Commit {cost_info["commit"]}, dirty={cost_info["dirty"]})')
print(f'MCA:    {MCA_CSV.relative_to(ROOT)}  (nur Dateidatum: {datetime.fromtimestamp(MCA_CSV.stat().st_mtime):%Y-%m-%d %H:%M})')
print(f'Zellen: {len(gdf_grid)}, davon zulässig (keine No-Go-Zone): {allowed.sum()}')

### C.2 Kennzahlen

Gerechnet wird nur auf zulässigen Zellen.

- **Spearman ρ**: Rangkorrelation von MCA-Score und Kosten. −1 heißt „beste MCA-Zellen sind die billigsten“.
- **Top-10 %-Überlappung**: Anteil der 10 % besten MCA-Zellen, die auch unter den 10 % billigsten sind. Bei Zufall liegt er bei etwa 10 %.
- **Pareto-Zellen**: Zellen, die keine andere Zelle zugleich bei MCA *und* Kosten schlägt. Das sind die eigentlichen Kandidaten.

In [ ]:
TOP_SHARE = 0.10


def pareto_cells(score, cost):
    # Nicht dominierte Zellen: aufsteigend nach Kosten, Zelle zählt, wenn ihr Score alle billigeren übertrifft.
    keep, best = [], -np.inf
    for i in cost.sort_values(kind='mergesort').index:
        if score[i] > best:
            keep.append(i)
            best = score[i]
    return keep


def compare_year(score, cost, mask, top_share=TOP_SHARE):
    score, cost = score[mask], cost[mask]
    n_top = round(top_share * len(score))
    top_mca, top_cost = set(score.nlargest(n_top).index), set(cost.nsmallest(n_top).index)
    return {
        'spearman_rho': spearmanr(score, cost).statistic,
        f'top{int(top_share*100)}_overlap': len(top_mca & top_cost) / n_top,
        f'top{int(top_share*100)}_jaccard': len(top_mca & top_cost) / len(top_mca | top_cost),
        'n_pareto': len(pareto_cells(score, cost)),
        'lcoh_median_top_mca': cost[list(top_mca)].median(),
        'lcoh_median_all': cost.median(),
    }


df_metrics = pd.DataFrame({year: compare_year(mca[year], lcoh[year], allowed) for year in YEARS})
df_metrics.round(3)

### C.3 Streudiagramm und Quadrantenkarte

Die Quadranten sind am Median der zulässigen Zellen geteilt.
Interessant sind die Nebendiagonalen: Zellen, die **geeignet, aber teuer** oder **günstig, aber weniger geeignet** sind. Dort widersprechen sich die Analysen.

In [ ]:
QUADRANTS = {
    'geeignet & günstig':         '#1b7837',
    'geeignet & teuer':           '#e08214',
    'weniger geeignet & günstig': '#4393c3',
    'weniger geeignet & teuer':   '#b2abd2',
    'No-Go':                      '#d9d9d9',
}


def quadrant(score, cost, mask):
    good  = score >= score[mask].median()
    cheap = cost  <= cost[mask].median()
    q = np.select([good & cheap, good & ~cheap, ~good & cheap], list(QUADRANTS)[:3], list(QUADRANTS)[3])
    return pd.Series(np.where(mask, q, 'No-Go'), index=score.index)


df_quadrant = pd.DataFrame({year: quadrant(mca[year], lcoh[year], allowed) for year in YEARS})

fig, axes = plt.subplots(2, len(YEARS), figsize=(13, 11), gridspec_kw={'height_ratios': [1, 1.3]})
for col, year in enumerate(YEARS):
    s, c, q = mca[year][allowed], lcoh[year][allowed], df_quadrant[year][allowed]
    ax = axes[0, col]
    ax.scatter(s, c, s=4, c=q.map(QUADRANTS), alpha=0.6, linewidths=0)
    p = pareto_cells(s, c)
    ax.plot(s[p], c[p], 'k.-', ms=6, lw=1, label=f'Pareto ({len(p)} Zellen)')
    ax.axvline(s.median(), color='grey', lw=0.8, ls='--')
    ax.axhline(c.median(), color='grey', lw=0.8, ls='--')
    ax.set(title=f'{year}: ρ = {df_metrics.loc["spearman_rho", year]:.2f}',
           xlabel='MCA-Score [0–100]', ylabel='LCOH [€/MWh$_{H_2}$]')
    ax.legend(loc='upper right', frameon=False)

    ax = axes[1, col]
    gdf_grid.plot(ax=ax, color=df_quadrant[year].map(QUADRANTS), linewidth=0)
    gdf_grid.iloc[p].boundary.plot(ax=ax, color='black', linewidth=0.8)
    ax.set_axis_off()
    ax.set_title(f'{year} (schwarz umrandet: Pareto-Zellen)')

fig.legend(handles=[Patch(color=v, label=k) for k, v in QUADRANTS.items()],
           loc='lower center', ncol=5, frameon=False)
fig.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

df_quadrant.apply(pd.Series.value_counts)

### C.4 Welche Kriterien tragen die Übereinstimmung?

Spearman-ρ zwischen den MCA-Einzelkriterien (Rohwerte, in MCA-Richtung: hoch = gut) und den Kostenanteilen, für 2025 und nur auf zulässigen Zellen.

- **Stark negativ** heißt: Das Kriterium bewertet dieselben Zellen gut, die in diesem Kostenanteil billig sind. Das kann eine Doppelerfassung sein, z. B. Wasser ↔ Wasserkosten.
- **Nahe 0** heißt: Das Kriterium bringt Information, die die Kosten nicht enthalten, oder umgekehrt.

Hinweise zu den Daten:
- Die Erreichbarkeit ist hier die ungewichtete Summe der Straßenklassen. Für die richtigen Gewichte wäre Umbau 2 nötig: Die Skripte müssten die skalierten Kriterien in den Laufordner schreiben.
- Elektrolyse- und RE-Stromkosten sind heute in allen Zellen gleich (Fehlerprotokoll Nr. 48). Die Kostenunterschiede kommen also nur aus Netz, Pipeline und Wasser.
- Die Kriterien sind die heutigen Dateien in `data/interim/mca_criteria/` und können zu einem anderen Stand gehören als `results/mca/`.

In [ ]:
CRIT = ROOT / 'data/interim/mca_criteria'
INVERTED = {'Landwirtschaft', 'Stadt'}          # in der MCA invertiert: hoher Anteil = schlecht
criteria_files = {
    'PV':             'results_pv_yeald.csv',
    'Wind':           'results_wind_flh.csv',
    'Erreichbarkeit': 'results_accessibility.csv',
    'Landwirtschaft': 'results_agriculture.csv',
    'Stadt':          'results_urban.csv',
    'Industrie':      'results_industrie.csv',
    'Ländlich':       'results_rural.csv',
    'Wasser':         'results_water_res_available.csv',
    'Nicht-Konflikt': 'results_non_conflict_areas.csv',
}
df_crit = pd.DataFrame({name: pd.read_csv(CRIT / f).sum(axis=1) for name, f in criteria_files.items()})
df_crit[list(INVERTED)] *= -1
df_crit['MCA gesamt'] = mca[2025]

cost_cols = {c.split(' [')[0].replace('_cost', '').replace('H2_Price_2025', 'LCOH gesamt'): c
             for c in df_cost[2025].columns if '[EUR/MWh_h2]' in c}
df_comp = pd.DataFrame({k: df_cost[2025][v] for k, v in cost_cols.items()})

# Konstante Anteile haben keine Rangkorrelation. Heute betrifft das Elektrolyse und RE-Strom:
# Alle Zellen beziehen vom selben besten Windstandort (Fehlerprotokoll Nr. 48).
constant = [k for k in df_comp if df_comp.loc[allowed, k].nunique() == 1]
if constant:
    print(f'In allen zulässigen Zellen gleich, daher weggelassen: {constant}')
    df_comp = df_comp.drop(columns=constant)

rho = pd.DataFrame({k: [spearmanr(df_crit.loc[allowed, crit], df_comp.loc[allowed, k]).statistic
                        for crit in df_crit] for k in df_comp}, index=df_crit.columns)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(rho.values, cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(rho.shape[1]), rho.columns, rotation=30, ha='right')
ax.set_yticks(range(rho.shape[0]), rho.index)
for (r, c), v in np.ndenumerate(rho.values):
    ax.text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=8,
            color='white' if abs(v) > 0.6 else 'black')
fig.colorbar(im, ax=ax, label='Spearman ρ  (negativ = stimmen überein)')
ax.set_title('MCA-Kriterien ↔ Kostenanteile 2025')
fig.tight_layout()
plt.show()

### C.5 Speichern (optional)

Mit `SAVE = True` landet der Vergleich in einem neuen Laufordner `results/runs/<datum>_<commit>_compare/`, inklusive Verweis auf die verwendeten Quellen.
Nach Umbau 1 wäre das ein Unterordner des gemeinsamen `_main`-Laufs.

In [ ]:
if SAVE:
    run_dir = new_run_dir('compare', note=NOTE, script='main.ipynb')
    df_metrics.to_csv(run_dir / 'compare_metrics.csv')
    df_quadrant.to_csv(run_dir / 'compare_quadrants.csv', index=False)
    rho.to_csv(run_dir / 'compare_criteria_spearman_2025.csv')
    (run_dir / 'compare_sources.json').write_text(json.dumps({
        'mca_csv': str(MCA_CSV.relative_to(ROOT)),
        'mca_mtime': datetime.fromtimestamp(MCA_CSV.stat().st_mtime).isoformat(timespec='minutes'),
        'cost_run': COST_RUN.name,
    }, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'Gespeichert: {run_dir.relative_to(ROOT)}')
else:
    print('SAVE = False: nichts gespeichert.')